# Generative AI Model Selection & Setup

### Model Selection & Setup


In this system, the Gemini API is integrated to support the generation of natural language explanations related to depression detection.  
The API is initialized using a secure API key stored in a .env file and loaded through environment variables to ensure that sensitive information is not exposed in the code.

The model **gemini-2.5-flash** was selected due to its ability to generate clear, context-aware, and human-readable responses.  
This is particularly important for a system that deals with sensitive topics such as mental health, where explanations must be understandable, supportive, and appropriately phrased.

The selected model provides a balance between response quality and system reliability, making it suitable for both development and testing.  
It allows the system to transform model predictions into meaningful explanations that can help users better understand their condition, while maintaining consistent performance

### API Key Management

The API key is stored in a local .env file to ensure that sensitive information is not exposed in the source code or shared in the repository.

The .env file contains environment variables, such as GEMINI_API_KEY, which are loaded at runtime using the python-dotenv library.  
This allows the system to securely access the API key without hardcoding it into the notebook.

The .env file is excluded from version control using .gitignore, ensuring that it is not uploaded to GitHub.  
Instead, a template file ( `.env.example`) can be shared with team members to guide them in setting up their own local environment variables.

This approach improves security and follows best practices for handling confidential data.

In [1]:
%pip install python-dotenv google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", gemini_api_key is not None)

client = genai.Client(api_key=gemini_api_key)

print("Gemini client loaded successfully")

API key loaded: True
Gemini client loaded successfully


## Prompt Template Design Documentation

The goal of the prompt engineering part is to design different prompt styles that convert the supervised model prediction into clear student lifestyle advice. The prediction output is based on the `Depression` target, where the student is classified as either at risk of depression or not at risk.

The prompts use the most relevant student lifestyle and academic features:

- CGPA
- Sleep Duration
- Study Hours
- Social Media Hours
- Physical Activity
- Stress Level
- Prediction Result

Four different prompt templates were designed to compare how prompt structure affects the quality of the generated advice.

## Prompt Templates

In [3]:
# ==============================
# Prompt Templates
# ==============================

def format_features(case):
    return f"""
CGPA: {case['cgpa']}
Sleep Duration: {case['sleep_duration']} hours
Study Hours: {case['study_hours']} hours
Social Media Hours: {case['social_media_hours']} hours
Physical Activity: {case['physical_activity']} hours
Stress Level: {case['stress_level']}
Prediction Result: {case['prediction']}
"""


template_1_basic = """
You are an assistant that provides simple student lifestyle advice.

Student information:
{features}

Based on the prediction result, explain the student's situation and give simple advice.

Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
Keep the response clear and easy to understand.
"""


template_2_structured = """
You are a mental health and academic advisor.

Student information:
{features}

Provide:
1. Explanation
2. Risk factors
3. Lifestyle recommendations
4. Study balance advice

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
"""


template_3_personalized = """
You are a supportive student coach.

Student information:
{features}

Give:
- Personalized advice
- Daily action steps
- Encouragement

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
"""


template_4_analytical = """
You are a data-driven advisor.

Student information:
{features}

Explain:
- Why this prediction happened
- Which factors affected it
- What should be improved first

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
"""

## Test Cases

In [4]:
test_cases = [
    {
        "case_id": "Case 1",
        "cgpa": 2.1,
        "sleep_duration": 4,
        "study_hours": 10,
        "social_media_hours": 5,
        "physical_activity": 0,
        "stress_level": 9,
        "prediction": "At risk of depression"
    },
    {
        "case_id": "Case 2",
        "cgpa": 3.5,
        "sleep_duration": 7,
        "study_hours": 5,
        "social_media_hours": 2,
        "physical_activity": 3,
        "stress_level": 3,
        "prediction": "Not at risk of depression"
    },
    {
        "case_id": "Case 3",
        "cgpa": 2.8,
        "sleep_duration": 5,
        "study_hours": 8,
        "social_media_hours": 4,
        "physical_activity": 1,
        "stress_level": 7,
        "prediction": "At risk of depression"
    }
]

## Prompt Preview

In [5]:
from IPython.display import display, Markdown

prompt_templates = {
    "Template 1": template_1_basic,
    "Template 2": template_2_structured,
    "Template 3": template_3_personalized,
    "Template 4": template_4_analytical
}

sample = format_features(test_cases[0])

for name, template in prompt_templates.items():
    display(Markdown(f"## {name}"))
    display(Markdown(template.format(features=sample)))

## Template 1


You are an assistant that provides simple student lifestyle advice.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Based on the prediction result, explain the student's situation and give simple advice.

Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.
Keep the response clear and easy to understand.


## Template 2


You are a mental health and academic advisor.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Provide:
1. Explanation
2. Risk factors
3. Lifestyle recommendations
4. Study balance advice

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.


## Template 3


You are a supportive student coach.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Give:
- Personalized advice
- Daily action steps
- Encouragement

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.


## Template 4


You are a data-driven advisor.

Student information:

CGPA: 2.1
Sleep Duration: 4 hours
Study Hours: 10 hours
Social Media Hours: 5 hours
Physical Activity: 0 hours
Stress Level: 9
Prediction Result: At risk of depression


Explain:
- Why this prediction happened
- Which factors affected it
- What should be improved first

Safety instruction: Do not claim that the student is clinically depressed. Treat the prediction as a risk indicator only.


# Implementation & API Integration Code 

In [6]:

def generate_ai_response(prompt):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Error: {e}"


## Prompt Execution

After designing the four prompt templates, each template was connected to the prepared test cases and executed using the Gemini API.

The purpose of this step is to generate sample outputs for each prompt template. These outputs will later be used by the output analysis to compare the templates based on relevance, clarity, personalization, completeness, and safety.



In [7]:
# ==============================
# Generate AI Outputs for All Templates (FINAL VERSION)
# ==============================

import pandas as pd
import os
import time

os.makedirs("Generative_AI/example_outputs", exist_ok=True)

results = []

def safe_generate(prompt, retries=3):
    for attempt in range(retries):
        response = generate_ai_response(prompt)

        # If success → return
        if not str(response).startswith("Error:"):
            return response

        # If quota error → wait and retry
        if "RESOURCE_EXHAUSTED" in str(response):
            print(f"Quota hit. Waiting 60s before retry ({attempt+1}/{retries})...")
            time.sleep(60)
        else:
            print("Other error:", response)
            return response

    return "Failed after retries"


for case in test_cases:
    features = format_features(case)

    for template_name, template in prompt_templates.items():
        final_prompt = template.format(features=features)

        print(f"Running: {case['case_id']} - {template_name}")

        ai_output = safe_generate(final_prompt)

        results.append({
            "case_id": case.get("case_id", "N/A"),
            "template_name": template_name,
            "prediction": case["prediction"],
            "prompt": final_prompt,
            "ai_output": ai_output
        })

        time.sleep(5)  # safe delay


outputs_df = pd.DataFrame(results)

outputs_df.to_csv(
    "Generative_AI/example_outputs/generated_ai_outputs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done. Outputs saved.")
outputs_df

Running: Case 1 - Template 1
Running: Case 1 - Template 2
Running: Case 1 - Template 3
Running: Case 1 - Template 4
Running: Case 2 - Template 1
Running: Case 2 - Template 2
Running: Case 2 - Template 3
Running: Case 2 - Template 4
Running: Case 3 - Template 1
Running: Case 3 - Template 2
Running: Case 3 - Template 3
Running: Case 3 - Template 4
Done. Outputs saved.


,case_id,template_name,prediction,prompt,ai_output
0,Case 1,Template 1,At risk of depression,\nYou are an assistant that provides simple st...,Hi there! Thanks for reaching out. Let's look ...
1,Case 1,Template 2,At risk of depression,\nYou are a mental health and academic advisor...,Hello! Thank you for reaching out and sharing ...
2,Case 1,Template 3,At risk of depression,\nYou are a supportive student coach.\n\nStude...,Hey there! Thanks for reaching out and being o...
3,Case 1,Template 4,At risk of depression,\nYou are a data-driven advisor.\n\nStudent in...,"As your data-driven advisor, let's break down ..."
4,Case 2,Template 1,Not at risk of depression,\nYou are an assistant that provides simple st...,Hi there!\n\nIt looks like you're doing a fant...
5,Case 2,Template 2,Not at risk of depression,\nYou are a mental health and academic advisor...,Hello there! I'm here to provide some insights...
6,Case 2,Template 3,Not at risk of depression,\nYou are a supportive student coach.\n\nStude...,Hey there! I'm so glad to connect with you. Lo...
7,Case 2,Template 4,Not at risk of depression,\nYou are a data-driven advisor.\n\nStudent in...,"Based on the data provided, here's an analysis..."
8,Case 3,Template 1,At risk of depression,\nYou are an assistant that provides simple st...,Hey there! Thanks for sharing your information...
9,Case 3,Template 2,At risk of depression,\nYou are a mental health and academic advisor...,Hello! Thank you for coming to speak with me t...


In [8]:
# ==============================
# Save Outputs as Markdown
# ==============================

md_path = "Generative_AI/example_outputs/generated_ai_outputs.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Generated AI Outputs\n\n")

    for _, row in outputs_df.iterrows():
        f.write(f"## {row['case_id']} - {row['template_name']}\n\n")
        f.write(f"**Prediction:** {row['prediction']}\n\n")
        f.write("### Prompt Used\n\n")
        f.write("```text\n")
        f.write(row["prompt"])
        f.write("\n```\n")
        f.write("### AI Output\n\n")
        f.write(row["ai_output"])
        f.write("\n\n---\n\n")

print(f"Saved markdown outputs to: {md_path}")

Saved markdown outputs to: Generative_AI/example_outputs/generated_ai_outputs.md


# Testing Framework & Output Comparison 

# Analysis: Qualitative & Quantitative Results

# Best Prompt Selection & Justification

# Integration Plan for Final System

# Ethical Considerations & Limitations 
